# Modul 4 · Cara AI Mendeteksi Keanehan Data (Anomali)
**Pelatihan Machine Learning & Data Analysis**

**Anomali** = data yang menyimpang jauh dari pola umum. Di dunia nyata bisa berarti:
- kesalahan input operator,
- transaksi mencurigakan / potensi fraud,
- kejadian luar biasa yang perlu ditindaklanjuti.

Mengecek ribuan transaksi satu per satu secara manual tidak mungkin. Di sinilah AI membantu: **AI menyaring otomatis, manusia tinggal memverifikasi.**

Tiga metode, dari yang paling sederhana sampai berbasis ML:

1. **Z-SCORE** — seberapa jauh nilai dari rata-rata (dalam satuan simpangan baku); |z| > 3 → patut dicurigai.
2. **IQR** — di luar "pagar" statistik kuartil (logika boxplot).
3. **ISOLATION FOREST** — algoritma ML yang "mengisolasi" titik aneh dengan melihat *beberapa kolom sekaligus*.

> ⚠️ Prasyarat: notebook `00_generate_dataset.ipynb` **dan** `01_pengolahan_data.ipynb` sudah dijalankan (modul ini memakai data bersih hasil Modul 1).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import IsolationForest

DIR = Path.cwd().parent

df = pd.read_csv(DIR / "dataset" / "data_transaksi_bersih.csv")
print(f"Data transaksi bersih: {len(df)} baris\n")

## METODE 1 : Z-SCORE per jenis part
*(harga Aki wajar berbeda dari harga Busi, jadi pembandingnya per kelompok)*

In [ ]:
def hitung_zscore(kelompok):
    return (kelompok - kelompok.mean()) / kelompok.std()

df["z_harga"] = df.groupby("nama_part")["harga_satuan"].transform(hitung_zscore)
df["z_qty"] = hitung_zscore(df["qty"])

anomali_z = df[(df["z_harga"].abs() > 3) | (df["z_qty"].abs() > 3)]
print(f"\nTransaksi dengan |z-score| > 3 : {len(anomali_z)} baris")
print(anomali_z[["id_transaksi", "dealer", "nama_part", "qty",
                 "harga_satuan", "z_harga", "z_qty"]]
      .round(1).head(10).to_string(index=False))

## METODE 2 : IQR (Interquartile Range - logika boxplot)

In [ ]:
q1, q3 = df["total"].quantile([0.25, 0.75])
iqr = q3 - q1
batas_bawah = q1 - 1.5 * iqr
batas_atas = q3 + 1.5 * iqr

anomali_iqr = df[(df["total"] < batas_bawah) | (df["total"] > batas_atas)]
print(f"\nBatas wajar nilai transaksi: Rp{batas_bawah:,.0f} s.d. Rp{batas_atas:,.0f}")
print(f"Transaksi di luar batas    : {len(anomali_iqr)} baris")
print("Catatan: IQR sederhana & cepat, tetapi pada data yang nilainya memang\n"
      "beragam (Busi murah vs Aki mahal) ia bisa terlalu banyak menandai.")

## METODE 3 : ISOLATION FOREST (machine learning, multi-kolom sekaligus)

In [ ]:
# Fitur yang dilihat bersamaan: qty, harga, dan total
fitur = df[["qty", "harga_satuan", "total"]]

# contamination = perkiraan proporsi anomali (di sini ~1%)
iso = IsolationForest(contamination=0.01, random_state=42)
df["flag_iso"] = iso.fit_predict(fitur)          # -1 = anomali, 1 = normal
df["skor_anomali"] = -iso.score_samples(fitur)   # makin besar = makin aneh

anomali_iso = (df[df["flag_iso"] == -1]
               .sort_values("skor_anomali", ascending=False))

print(f"\nIsolation Forest menandai {len(anomali_iso)} transaksi paling aneh:")
print(anomali_iso[["id_transaksi", "tanggal", "dealer", "nama_part",
                   "qty", "harga_satuan", "total", "skor_anomali"]]
      .round(2).to_string(index=False))

## INTERPRETASI: KENAPA TRANSAKSI INI DIANGGAP ANEH?

In [ ]:
harga_normal = df[df["flag_iso"] == 1].groupby("nama_part")["harga_satuan"].mean()
qty_normal = df[df["flag_iso"] == 1]["qty"].mean()

print()
for _, baris in anomali_iso.head(8).iterrows():
    alasan = []
    rasio_harga = baris["harga_satuan"] / harga_normal[baris["nama_part"]]
    if rasio_harga > 3:
        alasan.append(f"harga {rasio_harga:.0f}x lipat dari normal")
    elif rasio_harga < 0.3:
        alasan.append("harga jauh di bawah normal (potensi diskon tak sah)")
    if baris["qty"] > qty_normal * 5:
        alasan.append(f"qty {baris['qty']:.0f} (normal ~{qty_normal:.0f})")
    print(f"  {baris['id_transaksi']} | {baris['dealer']:<13} | "
          f"{baris['nama_part']:<12} -> {'; '.join(alasan) or 'pola tidak umum'}")

# Simpan daftar anomali untuk ditindaklanjuti tim terkait
file_laporan = DIR / "output" / "laporan_anomali_transaksi.csv"
anomali_iso.drop(columns=["z_harga", "z_qty", "flag_iso"]).to_csv(
    file_laporan, index=False)
print(f"\nLaporan anomali disimpan ke: {file_laporan.name}")

## VISUALISASI: normal vs anomali

In [ ]:
normal = df[df["flag_iso"] == 1]
plt.figure(figsize=(11, 5))
plt.scatter(normal["qty"], normal["harga_satuan"],
            s=14, alpha=0.35, color="#666666", label="Transaksi normal")
plt.scatter(anomali_iso["qty"], anomali_iso["harga_satuan"],
            s=70, color="#E60012", marker="x", label="Anomali (ditandai AI)")
plt.title("Deteksi Anomali Transaksi Sparepart - Isolation Forest")
plt.xlabel("Qty")
plt.ylabel("Harga satuan (Rp)")
plt.legend()
plt.tight_layout()
file_grafik = DIR / "output" / "modul4_deteksi_anomali.png"
plt.savefig(file_grafik, dpi=120)
plt.show()
print(f"Grafik disimpan ke: {file_grafik.name}")

print("""
--------------------------------------------------------------------------------
KESIMPULAN MODUL 4
--------------------------------------------------------------------------------
1. Mulailah dari metode sederhana (Z-score / IQR) - sering kali sudah cukup.
2. Isolation Forest unggul karena melihat beberapa kolom SEKALIGUS dan tidak
   perlu data berlabel "ini fraud / ini bukan".
3. AI berperan sebagai PENYARING; keputusan akhir tetap lewat verifikasi
   manusia. Outputnya adalah daftar prioritas untuk dicek, bukan vonis.
4. Pola yang sama bisa dipakai untuk: klaim garansi tak wajar, pemakaian
   sparepart bengkel, jam kerja mesin, kualitas hasil produksi, dll.
--------------------------------------------------------------------------------
""")

---
### 💡 Latihan mandiri
1. Ubah `contamination` Isolation Forest dari `0.01` menjadi `0.02` — berapa transaksi yang kini ditandai? Apa risikonya bila terlalu besar?
2. Turunkan ambang Z-score dari 3 menjadi 2,5. Berapa banyak "alarm" tambahan yang muncul, dan apakah semuanya benar-benar aneh?